# 4. Index Recipes into Qdrant

This notebook indexes the annotated recipe datasets into Qdrant using the backend indexing code. It supports both `dataset_10000_annotated.csv` and `dataset_full_annotated.csv`.

In [6]:
%pip install qdrant_client
%pip install sentence_transformers
%pip install pydantic_settings


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Setup

Run this from the repo root or from `notebooks/data_cleaning`. The notebook finds the repo root, loads `.env` if present, then imports the backend parser/vector store.

In [7]:
from __future__ import annotations

import csv
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start, *start.parents]
    for path in candidates:
        if (path / "backend" / "app").is_dir() and (path / "data").is_dir():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
BACKEND_ROOT = REPO_ROOT / "backend"

if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

load_dotenv(REPO_ROOT / ".env")

from app.config import settings  # noqa: E402
from app.models.recipe import RecipeDataPoint  # noqa: E402
from app.vector_db.parsing import (  # noqa: E402
    parse_int,
    parse_list,
    parse_normalized_ingredients,
)
from app.vector_db.recipe_vector_store import RecipeVectorStore  # noqa: E402

print(f"Repo root: {REPO_ROOT}")
print(f"Qdrant URL: {settings.qdrant_url}")
print(f"Embedding model: {settings.embedding_model}")

Repo root: /Users/jay/Desktop/dishify
Qdrant URL: http://localhost:6333
Embedding model: sentence-transformers/all-MiniLM-L6-v2


## Choose Dataset

Set `DATASET_KEY` to `"10000"` for the smaller annotated dataset or `"full"` for the full annotated dataset. Each dataset writes to a separate Qdrant collection by default.

In [ ]:
DATASETS = {
    "10000": {
        "csv": REPO_ROOT / "data" / "dataset_10000_annotated.csv",
        "collection": "recipes_10000",
    },
    "full": {
        "csv": REPO_ROOT / "data" / "dataset_full_annotated.csv",
        "collection": "recipes_full",
    },
}

DATASET_KEY = "full"  # Change to "full" to index the full annotated dataset.
DRY_RUN = False
DRY_RUN_LIMIT = 25
RECREATE_COLLECTION = True
QDRANT_BATCH_SIZE = 100
CSV_BATCH_SIZE = 5_000

dataset = DATASETS[DATASET_KEY]
csv_path = dataset["csv"]
collection_name = dataset["collection"]

if not csv_path.exists():
    raise FileNotFoundError(csv_path)

print(f"Dataset: {DATASET_KEY}")
print(f"CSV: {csv_path}")
print(f"Collection: {collection_name}")
print(f"Dry run: {DRY_RUN}")
print(f"Recreate collection: {RECREATE_COLLECTION}")

Dataset: 10000
CSV: /Users/jay/Desktop/dishify/data/dataset_10000_annotated.csv
Collection: recipes_10000
Dry run: True
Recreate collection: True


## Create Client and Load Model

In [9]:
client = QdrantClient(url=settings.qdrant_url)
model = SentenceTransformer(settings.embedding_model)

store = RecipeVectorStore(
    qdrant_client=client,
    embedding_model=model,
    collection_name=collection_name,
)

print(f"Vector size: {store.vector_size}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11716.57it/s]


Vector size: 384


/Users/jay/Desktop/dishify/backend/app/vector_db/recipe_vector_store.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.vector_size = embedding_model.get_sentence_embedding_dimension()


## Streaming CSV Parser

The full annotated dataset is several GB, so this notebook streams rows into indexing batches instead of loading the whole CSV into memory.

In [10]:
def recipe_from_row(row: dict[str, str]) -> RecipeDataPoint:
    ingredients = [
        str(item).strip()
        for item in parse_list(row["ingredients"])
        if item is not None and str(item).strip()
    ]
    raw_ingredients = [
        str(item).strip()
        for item in parse_list(row.get("raw_ingredients") or "")
        if item is not None and str(item).strip()
    ]
    if not raw_ingredients:
        raw_ingredients = list(ingredients)

    parsed_ingredients = parse_normalized_ingredients(
        row.get("normalized_ingredients") or ""
    )
    normalized_ingredients = [
        ingredient.name for ingredient in parsed_ingredients if ingredient.name
    ]

    return RecipeDataPoint(
        title=row["title"],
        ingredients=ingredients,
        raw_ingredients=raw_ingredients,
        parsed_ingredients=parsed_ingredients,
        normalized_ingredients=normalized_ingredients,
        directions=parse_list(row["directions"]),
        link=row["link"],
        source=row["source"],
        ner=parse_list(row.get("NER") or ""),
        exclusion_restrictions=parse_list(row.get("exclusion_restrictions") or ""),
        exclusion_restrictions_count=parse_int(row.get("exclusion_restrictions_count")),
    )


def iter_recipe_batches(path: Path, csv_batch_size: int):
    with path.open("r", encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file)
        batch: list[RecipeDataPoint] = []
        for row in reader:
            batch.append(recipe_from_row(row))
            if len(batch) >= csv_batch_size:
                yield batch
                batch = []
        if batch:
            yield batch


def count_csv_rows(path: Path) -> int:
    with path.open("r", encoding="utf-8", newline="") as file:
        return max(sum(1 for _ in file) - 1, 0)


with csv_path.open("r", encoding="utf-8", newline="") as file:
    sample = recipe_from_row(next(csv.DictReader(file)))

print(sample.title)
print(sample.raw_ingredients[:5])
print(sample.exclusion_restrictions[:10])

No-Bake Nut Cookies
['brown sugar', 'evaporated milk', 'vanilla', 'broken nuts', 'butter']
['nut_allergy', 'milk_allergy', 'alpha_gal_syndrome', 'lactose_intolerance', 'fodmap_intolerance', 'vegan', 'ovo_vegetarian', 'dairy_free', 'nut_free', 'low_fodmap']


## Index Selected Dataset

With `DRY_RUN = True`, this validates parsing and embeddings without creating a collection or upserting points. With `DRY_RUN = False`, it recreates the target collection when `RECREATE_COLLECTION = True`, then streams recipes from the CSV and upserts them in batches.

In [11]:
expected_rows = count_csv_rows(csv_path)
print(f"Expected rows: {expected_rows:,}")

if DRY_RUN:
    checked = 0
    for batch in iter_recipe_batches(csv_path, min(CSV_BATCH_SIZE, DRY_RUN_LIMIT)):
        for recipe in batch:
            text_for_embedding = f"""
            Title: {recipe.title}
            Title: {recipe.title}
            Raw ingredients: {", ".join(str(item) for item in recipe.raw_ingredients)}
            """
            vector = model.encode(text_for_embedding).tolist()
            checked += 1

            if checked <= 3:
                print(f"\nSample {checked}")
                print(f"Title: {recipe.title}")
                print(f"Vector dimensions: {len(vector)}")
                print(f"Raw ingredients: {recipe.raw_ingredients[:5]}")
                print(f"Exclusion restrictions: {recipe.exclusion_restrictions[:10]}")

            if checked >= DRY_RUN_LIMIT:
                break
        if checked >= DRY_RUN_LIMIT:
            break

    print(f"\nDry run complete. Parsed and embedded {checked:,} recipes.")
    print("No Qdrant collection was created or modified.")
else:
    store.create_collection(recreate=RECREATE_COLLECTION)

    indexed = 0
    for batch in iter_recipe_batches(csv_path, CSV_BATCH_SIZE):
        store.index_recipes(batch, batch_size=QDRANT_BATCH_SIZE, start_id=indexed)
        indexed += len(batch)
        print(f"Indexed {indexed:,}/{expected_rows:,}")

    collection_info = client.get_collection(collection_name)
    print(f"Indexed collection: {collection_name}")
    points_count = collection_info.points_count
    points_count_text = f"{points_count:,}" if points_count is not None else "unknown"
    print(f"Qdrant points count: {points_count_text}")

Expected rows: 10,000

Sample 1
Title: No-Bake Nut Cookies
Vector dimensions: 384
Raw ingredients: ['brown sugar', 'evaporated milk', 'vanilla', 'broken nuts', 'butter']
Exclusion restrictions: ['nut_allergy', 'milk_allergy', 'alpha_gal_syndrome', 'lactose_intolerance', 'fodmap_intolerance', 'vegan', 'ovo_vegetarian', 'dairy_free', 'nut_free', 'low_fodmap']

Sample 2
Title: Jewell Ball'S Chicken
Vector dimensions: 384
Raw ingredients: ['beef', 'chicken breasts', 'cream of mushroom soup', 'sour cream']
Exclusion restrictions: ['milk_allergy', 'alpha_gal_syndrome', 'lactose_intolerance', 'fodmap_intolerance', 'vegan', 'vegetarian', 'lacto_vegetarian', 'ovo_vegetarian', 'lacto_ovo_vegetarian', 'pescatarian']

Sample 3
Title: Creamy Corn
Vector dimensions: 384
Raw ingredients: ['pkg. frozen corn', 'pkg. cream cheese', 'butter', 'garlic powder', 'salt']
Exclusion restrictions: ['milk_allergy', 'corn_allergy', 'garlic_allergy', 'alpha_gal_syndrome', 'lactose_intolerance', 'fodmap_intolerance

## Retrieval Smoke Test

In [ ]:
if DRY_RUN:
    print("Skipping retrieval smoke test because DRY_RUN = True.")
else:
    results = store.retrieve_recipes(
        query="quick dinner with pasta",
        top_k=5,
        excluded_ingredients=["peanut", "shellfish"],
        available_ingredients=["pasta", "tomato", "garlic"],
    )

    for i, recipe in enumerate(results, start=1):
        print(f"{i}. {recipe.title}  score={recipe.score:.4f}")
        print(f"   restrictions={recipe.exclusion_restrictions}")
        print(f"   link={recipe.link}")

## Optional: Index Both Datasets

Run this cell only when you intentionally want to rebuild both collections. The full dataset can take a long time. This cell still streams rows, so it avoids loading the entire full CSV into memory.

In [ ]:
INDEX_BOTH = False

if INDEX_BOTH:
    for key, config in DATASETS.items():
        path = config["csv"]
        target_collection = config["collection"]
        if not path.exists():
            raise FileNotFoundError(path)

        expected = count_csv_rows(path)
        print(f"\nIndexing {key}: {path}")
        dataset_store = RecipeVectorStore(
            qdrant_client=client,
            embedding_model=model,
            collection_name=target_collection,
        )

        print(f"Recreating {target_collection}")
        dataset_store.create_collection(recreate=True)
        indexed = 0
        for batch in iter_recipe_batches(path, CSV_BATCH_SIZE):
            dataset_store.index_recipes(
                batch, batch_size=QDRANT_BATCH_SIZE, start_id=indexed
            )
            indexed += len(batch)
            print(f"Indexed {indexed:,}/{expected:,}")

        info = client.get_collection(target_collection)
        points_count = info.points_count
        points_count_text = (
            f"{points_count:,}" if points_count is not None else "unknown"
        )
        print(f"Done: {target_collection} has {points_count_text} points")